# Nemotron GRPO + PRM — Reinforcement Learning Notebook

**GRPO (Group Relative Policy Optimization) + PRM (Process Reward Model)**
for NVIDIA Nemotron reasoning competition.

**Architecture:** vLLM fast generation (~2500 tok/s) + HF PEFT training (alternating)

**Pipeline:** SFT warm-up adapter → GRPO rounds (generate→score→train→sync) → RELEX extrapolation

**Output:** `adapter_config.json` + `adapter_model.safetensors`

In [ ]:
# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================
import subprocess, sys

def pip_install(*pkgs):
    for pkg in pkgs:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg],
                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Core RL + generation stack
pip_install("trl>=0.17.0", "peft>=0.15.0", "accelerate", "bitsandbytes")

# vLLM for fast rollout generation
pip_install("vllm>=0.8.0")

print("[ok] All dependencies installed")

In [ ]:
# ============================================================
# 2. IMPORTS & ENVIRONMENT
# ============================================================
import os, sys, json, re, time, gc, copy, random, math
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from collections import defaultdict
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel, TaskType

# Environment
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name} | VRAM: {gpu.total_mem / 1e9:.1f} GB")

In [ ]:
# ============================================================
# 3. HYPERPARAMETERS
# ============================================================

# ── Model paths ──────────────────────────────────────────────
MODEL_PATH = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"
# If you have an SFT adapter from Phase 1, set this path:
SFT_ADAPTER_PATH = None   # e.g. "/kaggle/input/datasets/.../sft_adapter"
OUTPUT_DIR        = "/kaggle/working/grpo_adapter"
BEST_ADAPTER_DIR  = "/kaggle/working/best_adapter"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(BEST_ADAPTER_DIR, exist_ok=True)

# ── LoRA config (MUST match SFT adapter if resuming) ─────────
LORA_RANK    = 32
LORA_ALPHA   = 64       # proven config: alpha=64, LR=5e-5
LORA_DROPOUT = 0.0
MOE_LORA_MODE = "tied"  # tied MoE experts (same as SFT notebook)

# ── GRPO hyperparameters ─────────────────────────────────────
GROUP_SIZE          = 4       # rollouts per prompt (vLLM makes this fast)
CLIP_EPSILON        = 0.2     # PPO-style clipping
KL_COEFF            = 0.01    # KL penalty coefficient
GRPO_LR             = 5e-6    # 10x lower than SFT
MAX_NEW_TOKENS      = 6144    # gold CoTs max ~3800; exploration headroom
TEMPERATURE         = 0.7     # encourage exploration
TOP_P               = 0.95
NUM_ROUNDS          = 10      # generate-train rounds
PROMPTS_PER_ROUND   = 64      # prompts sampled each round
SFT_STEPS_PER_ROUND = 8       # SFT regularization steps per round
GRPO_EPOCHS_PER_ROUND = 1     # policy update epochs per round
GRAD_ACCUM_STEPS    = 8       # gradient accumulation for GRPO updates
MAX_GRAD_NORM       = 1.0

# ── Reward weights ───────────────────────────────────────────
ORM_WEIGHT = 0.7    # outcome (correctness)
PRM_WEIGHT = 0.3    # process (reasoning quality)

# ── Data ─────────────────────────────────────────────────────
CATEGORY_FILES = [
    "train_cot_bit_manipulation.jsonl",
    "train_cot_cipher.jsonl",
    "train_cot_cryptarithm_deduce.jsonl",
    "train_cot_cryptarithm_guess.jsonl",
    "train_cot_equation_numeric_deduce.jsonl",
    "train_cot_equation_numeric_guess.jsonl",
    "train_cot_gravity.jsonl",
    "train_cot_numeral.jsonl",
    "train_cot_unit_conversion.jsonl",
]

DATA_DIR_CANDIDATES = [
    "/kaggle/input/datasets/asharamkanderiwal/nvidia-dataset/all_categorical_splits",
    str(Path.cwd().parent / "data" / "processed" / "all_categorical_splits"),
    str(Path.cwd() / "all_categorical_splits_v14"),
]

# ── vLLM config ──────────────────────────────────────────────
VLLM_GPU_UTIL = 0.88   # leave headroom for CUDA overhead
VLLM_MAX_MODEL_LEN = 8192

print("=" * 60)
print("  GRPO + PRM CONFIG")
print("=" * 60)
print(f"  LoRA          : r={LORA_RANK}, alpha={LORA_ALPHA}")
print(f"  Group size    : {GROUP_SIZE}")
print(f"  Clip epsilon  : {CLIP_EPSILON}")
print(f"  KL coeff      : {KL_COEFF}")
print(f"  GRPO LR       : {GRPO_LR}")
print(f"  Max new tokens: {MAX_NEW_TOKENS}")
print(f"  Temperature   : {TEMPERATURE}")
print(f"  Rounds        : {NUM_ROUNDS}")
print(f"  Prompts/round : {PROMPTS_PER_ROUND}")
print(f"  Reward weights: ORM={ORM_WEIGHT}, PRM={PRM_WEIGHT}")

In [ ]:
# ============================================================
# 4. LOAD DATA — extract (prompt, ground_truth) pairs
# ============================================================

def extract_boxed_answer(text: str) -> Optional[str]:
    """Extract the LAST \\boxed{...} answer from text, handling nested braces."""
    # Find all \boxed{...} patterns
    results = []
    i = 0
    while i < len(text):
        idx = text.find("\\boxed{", i)
        if idx == -1:
            break
        # Find matching closing brace
        depth = 0
        start = idx + len("\\boxed{")
        for j in range(start, len(text)):
            if text[j] == '{':
                depth += 1
            elif text[j] == '}':
                if depth == 0:
                    results.append(text[start:j])
                    break
                depth -= 1
        i = start
    return results[-1].strip() if results else None

# Find data directory
data_dir = None
for cand in DATA_DIR_CANDIDATES:
    if os.path.isdir(cand):
        data_dir = cand
        break
assert data_dir is not None, f"Data dir not found in: {DATA_DIR_CANDIDATES}"
print(f"Data dir: {data_dir}")

# Load all samples
all_samples = []
category_counts = defaultdict(int)

for fname in CATEGORY_FILES:
    fpath = os.path.join(data_dir, fname)
    if not os.path.exists(fpath):
        print(f"  [warn] Missing: {fname}")
        continue
    with open(fpath) as f:
        for line in f:
            d = json.loads(line)
            cat = d["category"]
            prompt = d["messages"][0]["content"]
            gold_response = d["messages"][1]["content"]
            ground_truth = extract_boxed_answer(gold_response)

            if ground_truth is None:
                continue  # skip samples without parseable answer

            all_samples.append({
                "category": cat,
                "prompt": prompt,
                "gold_response": gold_response,
                "ground_truth": ground_truth,
            })
            category_counts[cat] += 1

print(f"\nLoaded {len(all_samples)} samples:")
for cat, n in sorted(category_counts.items()):
    print(f"  {cat:35s} {n:5d}")

# Shuffle for training
random.seed(42)
random.shuffle(all_samples)

In [ ]:
# ============================================================
# 5. REWARD FUNCTIONS — ORM + PRM
# ============================================================

def compute_orm_reward(generated_text: str, ground_truth: str) -> float:
    """Outcome Reward: does \\boxed{answer} match ground truth?"""
    pred = extract_boxed_answer(generated_text)
    if pred is None:
        return -1.0    # no answer extracted → strong penalty
    # Normalize whitespace for comparison
    pred_clean = pred.strip().lower()
    gt_clean = ground_truth.strip().lower()
    if pred_clean == gt_clean:
        return 1.0     # correct!
    return -0.5        # wrong answer (less harsh than no-answer)


def compute_prm_reward(generated_text: str) -> float:
    """Process Reward: evaluate reasoning quality heuristically."""
    score = 0.0

    # 1. Format: proper <think>...</think> structure
    has_think_open = "<think>" in generated_text
    has_think_close = "</think>" in generated_text
    if has_think_open and has_think_close:
        score += 0.10
    elif has_think_open:
        score += 0.03   # partial credit

    # 2. Has \boxed{} answer tag
    if "\\boxed{" in generated_text:
        score += 0.10

    # 3. Step structure: numbered steps, bullet points, or "Step N" markers
    step_patterns = [
        r"Step \d",          # "Step 1", "Step 2"
        r"step \d",
        r"^\d+\.",          # "1.", "2." at line start
        r"^\d+\)",          # "1)", "2)"
        r"^- ",              # bullet points
    ]
    step_count = 0
    for pat in step_patterns:
        step_count += len(re.findall(pat, generated_text, re.MULTILINE))
    if step_count >= 3:
        score += 0.10
    elif step_count >= 1:
        score += 0.05

    # 4. Length appropriateness (not too short or too long)
    text_len = len(generated_text)
    if 200 < text_len < 15000:
        score += 0.05
    elif text_len < 50:
        score -= 0.05    # way too short, probably garbage

    # 5. Verification attempt: model self-checks
    verify_patterns = [
        r"verif", r"check", r"confirm", r"validate",
        r"[✓✗]", r"\[ok\]", r"correct",
        r"All constraints satisfied",
    ]
    for pat in verify_patterns:
        if re.search(pat, generated_text, re.IGNORECASE):
            score += 0.05
            break

    return min(score, 0.40)   # cap PRM at 0.40


def compute_reward(generated_text: str, ground_truth: str) -> float:
    """Combined ORM + PRM reward."""
    orm = compute_orm_reward(generated_text, ground_truth)
    prm = compute_prm_reward(generated_text)
    return ORM_WEIGHT * orm + PRM_WEIGHT * prm


# ── Quick sanity test ────────────────────────────────────────
_test_good = "<think>\nStep 1: Analyze...\nStep 2: Compute...\nStep 3: Verify...\nAll constraints satisfied.\n</think>\n\\boxed{42}"
_test_bad  = "I don't know"
_test_wrong = "<think>\nLet me try...\n</think>\n\\boxed{WRONG}"

print("Reward sanity check:")
print(f"  Good answer (gt=42):  {compute_reward(_test_good, '42'):.3f}")
print(f"  Wrong answer (gt=42): {compute_reward(_test_wrong, '42'):.3f}")
print(f"  No answer (gt=42):    {compute_reward(_test_bad, '42'):.3f}")

In [ ]:
# ============================================================
# 6. vLLM GENERATION HELPER
# ============================================================

def load_tokenizer(model_path: str):
    """Load tokenizer (shared between vLLM and HF)."""
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def format_prompt_for_generation(prompt: str, tokenizer) -> str:
    """Apply chat template to a user prompt."""
    messages = [{"role": "user", "content": prompt}]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def generate_rollouts_vllm(
    model_path: str,
    lora_path: Optional[str],
    prompts: List[str],
    tokenizer,
    group_size: int = 4,
    max_tokens: int = 6144,
    temperature: float = 0.7,
    top_p: float = 0.95,
):
    """
    Generate G rollouts per prompt using vLLM.
    Returns list of list of dicts: rollouts[prompt_idx][group_idx] = {text, logprobs}
    """
    from vllm import LLM, SamplingParams
    from vllm.lora.request import LoRARequest

    print(f"  [vLLM] Loading engine (lora={lora_path is not None})...")
    t0 = time.time()

    engine_kwargs = dict(
        model=model_path,
        dtype="bfloat16",
        max_model_len=VLLM_MAX_MODEL_LEN,
        trust_remote_code=True,
        gpu_memory_utilization=VLLM_GPU_UTIL,
        enforce_eager=True,       # save memory on Blackwell
    )
    if lora_path and os.path.exists(os.path.join(lora_path, "adapter_model.safetensors")):
        engine_kwargs["enable_lora"] = True
        engine_kwargs["max_lora_rank"] = LORA_RANK

    engine = LLM(**engine_kwargs)
    print(f"  [vLLM] Engine loaded in {time.time()-t0:.1f}s")

    # Format prompts
    formatted = [format_prompt_for_generation(p, tokenizer) for p in prompts]

    sampling_params = SamplingParams(
        n=group_size,
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
        logprobs=1,               # per-token logprobs for GRPO
        stop=["<|endoftext|>"],
    )

    lora_request = None
    if lora_path and os.path.exists(os.path.join(lora_path, "adapter_model.safetensors")):
        lora_request = LoRARequest("grpo_lora", 1, lora_path)

    print(f"  [vLLM] Generating {len(formatted)} x {group_size} = {len(formatted)*group_size} completions...")
    t1 = time.time()
    outputs = engine.generate(formatted, sampling_params, lora_request=lora_request)
    gen_time = time.time() - t1
    total_tokens = sum(len(c.token_ids) for o in outputs for c in o.outputs)
    print(f"  [vLLM] Generated {total_tokens} tokens in {gen_time:.1f}s ({total_tokens/gen_time:.0f} tok/s)")

    # Parse outputs
    rollouts = []
    for output in outputs:
        group = []
        for completion in output.outputs:
            # Extract per-token logprobs
            token_lps = []
            if completion.logprobs:
                for lp_dict in completion.logprobs:
                    if lp_dict:
                        # logprobs is a list of dicts {token_id: Logprob}
                        # Get the logprob of the chosen token
                        vals = list(lp_dict.values())
                        token_lps.append(vals[0].logprob if vals else 0.0)
                    else:
                        token_lps.append(0.0)

            group.append({
                "text": completion.text,
                "token_ids": list(completion.token_ids),
                "token_logprobs": token_lps,
                "cumulative_logprob": completion.cumulative_logprob,
            })
        rollouts.append(group)

    # Free vLLM memory
    del engine
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  [vLLM] Engine freed, VRAM reclaimed")

    return rollouts

In [ ]:
# ============================================================
# 7. HF MODEL LOADING FOR TRAINING
# ============================================================

def get_lora_target_modules(model) -> list:
    """Detect LoRA target modules (same logic as SFT notebook)."""
    from collections import Counter
    linear_suffixes = Counter()
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            suffix = name.split(".")[-1]
            linear_suffixes[suffix] += 1

    ATTENTION = ["q_proj", "k_proj", "v_proj", "o_proj"]
    MAMBA     = ["in_proj", "out_proj"]
    MLP_UP    = ["up_proj"]
    MLP_DOWN  = ["down_proj"]
    EXCLUDE   = {"lm_head", "embed_tokens", "shared", "router", "score", "classifier"}

    targets = []
    for group in [ATTENTION, MAMBA, MLP_UP, MLP_DOWN]:
        for name in group:
            if name in linear_suffixes and name not in EXCLUDE:
                targets.append(name)
    return targets


def load_hf_model_with_lora(model_path: str, lora_path: Optional[str] = None):
    """Load HF model with LoRA for training."""
    print(f"  [HF] Loading base model in bf16...")
    t0 = time.time()

    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map={"": 0},
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
        attn_implementation="eager",
    )
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )

    if lora_path and os.path.exists(os.path.join(lora_path, "adapter_config.json")):
        # Resume from existing adapter
        print(f"  [HF] Loading existing LoRA adapter from {lora_path}")
        model = PeftModel.from_pretrained(model, lora_path, is_trainable=True)
    else:
        # Fresh LoRA
        print(f"  [HF] Applying fresh LoRA (r={LORA_RANK}, alpha={LORA_ALPHA})")
        targets = get_lora_target_modules(model)
        print(f"  [HF] Target modules: {targets}")
        lora_config = LoraConfig(
            r=LORA_RANK,
            lora_alpha=LORA_ALPHA,
            target_modules=targets,
            lora_dropout=LORA_DROPOUT,
            bias="none",
            task_type=TaskType.CAUSAL_LM,
        )
        model = get_peft_model(model, lora_config)

    model.enable_input_require_grads()

    # Cast LoRA params to fp32 for numerical stability
    for name, param in model.named_parameters():
        if ".lora_" in name:
            param.data = param.data.to(torch.float32)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"  [HF] Model loaded in {time.time()-t0:.1f}s")
    print(f"  [HF] Trainable: {trainable/1e6:.1f}M / {total/1e6:.0f}M total")
    print(f"  [HF] VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

    return model


def free_hf_model(model):
    """Free HF model and reclaim VRAM."""
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  [HF] Model freed, VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ============================================================
# 8. GRPO TRAINING CORE
# ============================================================

def compute_per_token_logprobs_batch(
    model, tokenizer, prompt_texts: List[str], response_texts: List[str]
) -> List[torch.Tensor]:
    """
    Compute per-token log-probs for each (prompt, response) pair.
    Returns list of 1D tensors, one per sample (response tokens only).
    Processes one sample at a time for memory safety on 30B model.
    """
    results = []
    for prompt, response in zip(prompt_texts, response_texts):
        full_text = prompt + response
        inputs = tokenizer(
            full_text, return_tensors="pt",
            truncation=True, max_length=VLLM_MAX_MODEL_LEN,
        ).to(model.device)

        prompt_ids = tokenizer(prompt, return_tensors="pt")
        prompt_len = prompt_ids.input_ids.shape[1]
        total_len = inputs.input_ids.shape[1]

        if total_len <= prompt_len + 1:
            results.append(torch.zeros(1, device=model.device))
            continue

        outputs = model(**inputs)
        logits = outputs.logits

        # Shift: logits[t] predicts token[t+1]
        shift_logits = logits[:, prompt_len - 1 : total_len - 1, :]
        shift_labels = inputs.input_ids[:, prompt_len:]

        log_probs = F.log_softmax(shift_logits.float(), dim=-1)
        per_token = log_probs.gather(-1, shift_labels.unsqueeze(-1)).squeeze(-1)

        results.append(per_token.squeeze(0).detach())

    return results


@torch.no_grad()
def compute_ref_and_old_logprobs(
    model, tokenizer, prompt_texts, response_texts
) -> Tuple[List[torch.Tensor], List[torch.Tensor]]:
    """
    Compute reference logprobs (LoRA disabled) and old-policy logprobs (LoRA enabled).
    Done without gradients for efficiency.
    """
    # Old policy logprobs (current LoRA weights)
    model.eval()
    old_lps = compute_per_token_logprobs_batch(model, tokenizer, prompt_texts, response_texts)

    # Reference logprobs (LoRA disabled = base model)
    model.disable_adapter_layers()
    ref_lps = compute_per_token_logprobs_batch(model, tokenizer, prompt_texts, response_texts)
    model.enable_adapter_layers()

    return ref_lps, old_lps


def grpo_policy_loss(
    new_logprobs: torch.Tensor,    # [T] per-token log-probs under current policy
    old_logprobs: torch.Tensor,    # [T] per-token log-probs under old policy
    ref_logprobs: torch.Tensor,    # [T] per-token log-probs under reference
    advantage: float,              # scalar group-relative advantage
    clip_eps: float = 0.2,
    kl_coeff: float = 0.01,
) -> torch.Tensor:
    """Compute clipped GRPO loss + KL penalty for one sample."""
    # Truncate to minimum length
    min_len = min(len(new_logprobs), len(old_logprobs), len(ref_logprobs))
    if min_len == 0:
        return torch.tensor(0.0, device=new_logprobs.device, requires_grad=True)

    new_lp = new_logprobs[:min_len]
    old_lp = old_logprobs[:min_len]
    ref_lp = ref_logprobs[:min_len]

    # Per-token ratio
    log_ratio = new_lp - old_lp
    ratio = torch.exp(log_ratio)

    # Clipped surrogate
    adv = torch.tensor(advantage, device=new_lp.device, dtype=new_lp.dtype)
    surr1 = ratio * adv
    surr2 = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * adv
    policy_loss = -torch.min(surr1, surr2).mean()

    # KL penalty (per-token, against reference)
    kl = (new_lp - ref_lp).mean()
    kl_penalty = kl_coeff * kl

    return policy_loss + kl_penalty


def grpo_training_step(
    model, tokenizer, optimizer,
    prompt_texts: List[str],
    rollout_groups: List[List[dict]],
    rewards_groups: List[List[float]],
    ref_logprobs_groups: List[List[torch.Tensor]],
    old_logprobs_groups: List[List[torch.Tensor]],
    clip_eps: float = 0.2,
    kl_coeff: float = 0.01,
) -> dict:
    """
    One GRPO gradient step over all prompts and their rollout groups.
    Uses gradient accumulation to handle memory constraints.
    """
    model.train()
    optimizer.zero_grad()

    total_loss = 0.0
    total_policy_loss = 0.0
    n_samples = 0

    for i, (prompt, rollouts, rewards, ref_lps, old_lps) in enumerate(
        zip(prompt_texts, rollout_groups, rewards_groups,
            ref_logprobs_groups, old_logprobs_groups)
    ):
        # Group-relative advantages
        rewards_arr = np.array(rewards)
        mean_r = rewards_arr.mean()
        std_r = rewards_arr.std() + 1e-8
        advantages = (rewards_arr - mean_r) / std_r

        formatted_prompt = format_prompt_for_generation(prompt, tokenizer)

        for j, (rollout, adv, ref_lp, old_lp) in enumerate(
            zip(rollouts, advantages, ref_lps, old_lps)
        ):
            # Compute new logprobs (WITH gradients)
            new_lps = compute_per_token_logprobs_batch(
                model, tokenizer,
                [formatted_prompt], [rollout["text"]]
            )
            new_lp = new_lps[0]

            # Compute loss
            loss = grpo_policy_loss(
                new_lp, old_lp, ref_lp, float(adv), clip_eps, kl_coeff
            )

            # Scale for gradient accumulation
            scaled_loss = loss / (len(prompt_texts) * GROUP_SIZE)
            scaled_loss.backward()

            total_loss += loss.item()
            n_samples += 1

    # Gradient step
    torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad],
        max_norm=MAX_GRAD_NORM,
    )
    optimizer.step()
    optimizer.zero_grad()

    return {
        "loss": total_loss / max(n_samples, 1),
        "n_samples": n_samples,
    }

In [ ]:
# ============================================================
# 9. SFT REGULARIZATION STEP
# ============================================================

def sft_step(model, tokenizer, optimizer, samples: List[dict], max_seq_len: int = 8192):
    """
    Standard SFT step on gold CoT data (regularization to prevent format drift).
    """
    model.train()
    optimizer.zero_grad()
    total_loss = 0.0
    n = 0

    for sample in samples:
        messages = [
            {"role": "user", "content": sample["prompt"]},
            {"role": "assistant", "content": sample["gold_response"]},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False)
        inputs = tokenizer(
            text, return_tensors="pt",
            truncation=True, max_length=max_seq_len,
        ).to(model.device)

        # Mask prompt tokens (only compute loss on assistant response)
        prompt_only = tokenizer.apply_chat_template(
            [{"role": "user", "content": sample["prompt"]}],
            tokenize=False, add_generation_prompt=True,
        )
        prompt_len = len(tokenizer(prompt_only).input_ids)

        labels = inputs.input_ids.clone()
        labels[:, :prompt_len] = -100  # mask prompt

        outputs = model(**inputs, labels=labels)
        loss = outputs.loss / len(samples)
        loss.backward()

        total_loss += outputs.loss.item()
        n += 1

    torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad],
        max_norm=MAX_GRAD_NORM,
    )
    optimizer.step()
    optimizer.zero_grad()

    return total_loss / max(n, 1)

In [ ]:
# ============================================================
# 10. RELEX EXTRAPOLATION (Rank-1 SVD + Linear Fit)
# ============================================================
# From paper: "You Only Need Minimal RLVR Training" (2605.21468)
# RL weight deltas are rank-1 and linear → extrapolate from short training

class RELEXTracker:
    """
    Track LoRA weight snapshots across GRPO rounds for RELEX extrapolation.
    Stores flattened deltas in memory (NOT on disk — Kaggle 20GB limit).
    """
    def __init__(self):
        self.base_state = None      # LoRA weights at round 0
        self.snapshots = []         # list of (round_idx, state_dict)

    def set_base(self, model):
        """Capture base LoRA weights (before any GRPO training)."""
        self.base_state = {
            name: param.detach().cpu().clone()
            for name, param in model.named_parameters()
            if param.requires_grad and ".lora_" in name
        }
        print(f"  [RELEX] Base snapshot: {len(self.base_state)} LoRA params")

    def snapshot(self, model, round_idx: int):
        """Capture current LoRA weights."""
        state = {
            name: param.detach().cpu().clone()
            for name, param in model.named_parameters()
            if param.requires_grad and ".lora_" in name
        }
        self.snapshots.append((round_idx, state))
        print(f"  [RELEX] Snapshot at round {round_idx} ({len(self.snapshots)} total)")

    def extrapolate(self, target_round: int) -> Optional[dict]:
        """
        Apply RELEX: rank-1 SVD on LoRA weight deltas, linear extrapolation.
        Returns predicted LoRA state_dict at target_round.
        """
        if self.base_state is None or len(self.snapshots) < 3:
            print("  [RELEX] Not enough snapshots (need >= 3). Skipping.")
            return None

        print(f"  [RELEX] Extrapolating to round {target_round} from {len(self.snapshots)} snapshots...")
        predicted_state = {}

        for param_name in self.base_state:
            base_w = self.base_state[param_name]
            base_flat = base_w.flatten().float()

            # Build trajectory matrix: rows = snapshots, cols = flattened param deltas
            deltas = []
            rounds = []
            for ridx, state in self.snapshots:
                delta = state[param_name].flatten().float() - base_flat
                deltas.append(delta)
                rounds.append(ridx)

            if len(deltas) < 2:
                predicted_state[param_name] = self.snapshots[-1][1][param_name]
                continue

            M = torch.stack(deltas, dim=0)  # [T, D]

            # Rank-1 SVD
            try:
                U, S, Vt = torch.linalg.svd(M, full_matrices=False)
            except Exception:
                predicted_state[param_name] = self.snapshots[-1][1][param_name]
                continue

            v1 = Vt[0]               # top-1 right singular vector [D]
            c1 = U[:, 0] * S[0]      # rank-1 coefficients [T]

            # Linear fit: c(t) = a*t + b
            t = torch.tensor(rounds, dtype=torch.float32)
            if t.std() < 1e-8:
                predicted_state[param_name] = self.snapshots[-1][1][param_name]
                continue

            a = ((t * c1).mean() - t.mean() * c1.mean()) / (t.var() + 1e-8)
            b = c1.mean() - a * t.mean()

            # Extrapolate
            c_target = a * target_round + b
            delta_target = c_target * v1
            predicted_w = base_flat + delta_target

            predicted_state[param_name] = predicted_w.reshape(base_w.shape).to(base_w.dtype)

        print(f"  [RELEX] Extrapolation complete for {len(predicted_state)} params")
        return predicted_state

    def apply_to_model(self, model, predicted_state: dict):
        """Load predicted weights into model."""
        with torch.no_grad():
            for name, param in model.named_parameters():
                if name in predicted_state:
                    param.data.copy_(predicted_state[name].to(param.device))
        print(f"  [RELEX] Applied extrapolated weights to model")


relex = RELEXTracker()

In [ ]:
# ============================================================
# 11. CHECKPOINT MANAGER — best-only (20GB Kaggle limit)
# ============================================================

class CheckpointManager:
    """Keep only the best adapter checkpoint to stay within 20GB output."""

    def __init__(self, save_dir: str):
        self.save_dir = save_dir
        self.best_reward = float("-inf")
        self.best_round = -1
        os.makedirs(save_dir, exist_ok=True)

    def maybe_save(self, model, mean_reward: float, round_idx: int) -> bool:
        """Save if this is the best round so far. Returns True if saved."""
        if mean_reward > self.best_reward:
            self.best_reward = mean_reward
            self.best_round = round_idx
            # Overwrite previous best
            model.save_pretrained(self.save_dir)
            size_mb = sum(
                os.path.getsize(os.path.join(self.save_dir, f))
                for f in os.listdir(self.save_dir)
            ) / 1e6
            print(f"  [CKPT] NEW BEST at round {round_idx}: reward={mean_reward:.4f} ({size_mb:.1f} MB)")
            return True
        else:
            print(f"  [CKPT] Round {round_idx} reward={mean_reward:.4f} < best={self.best_reward:.4f} — skipped")
            return False

    def status(self) -> str:
        return f"Best: round {self.best_round}, reward={self.best_reward:.4f}"


ckpt_mgr = CheckpointManager(BEST_ADAPTER_DIR)

In [ ]:
# ============================================================
# 12. MAIN TRAINING LOOP — Generate → Score → Train → Sync
# ============================================================

tokenizer = load_tokenizer(MODEL_PATH)
print(f"Tokenizer loaded: vocab_size={tokenizer.vocab_size}")

# ── Training stats ───────────────────────────────────────────
stats_log = []

for round_idx in range(NUM_ROUNDS):
    t_round = time.time()
    print(f"\n{'='*60}")
    print(f"  ROUND {round_idx + 1} / {NUM_ROUNDS}")
    print(f"{'='*60}")

    # ── Sample prompts for this round ────────────────────────
    round_samples = random.sample(all_samples, min(PROMPTS_PER_ROUND, len(all_samples)))
    prompts = [s["prompt"] for s in round_samples]
    ground_truths = [s["ground_truth"] for s in round_samples]

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # PHASE A: GENERATE ROLLOUTS WITH vLLM
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    print(f"\n[Phase A] Generating rollouts with vLLM...")
    lora_for_gen = BEST_ADAPTER_DIR if round_idx > 0 else SFT_ADAPTER_PATH

    rollouts = generate_rollouts_vllm(
        model_path=MODEL_PATH,
        lora_path=lora_for_gen,
        prompts=prompts,
        tokenizer=tokenizer,
        group_size=GROUP_SIZE,
        max_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        top_p=TOP_P,
    )

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # PHASE B: SCORE WITH ORM + PRM
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    print(f"\n[Phase B] Computing rewards...")
    rewards_groups = []
    all_rewards = []
    n_correct = 0
    n_total = 0

    for i, (group, gt) in enumerate(zip(rollouts, ground_truths)):
        group_rewards = []
        for rollout in group:
            r = compute_reward(rollout["text"], gt)
            group_rewards.append(r)
            all_rewards.append(r)
            n_total += 1
            if compute_orm_reward(rollout["text"], gt) > 0.5:
                n_correct += 1
        rewards_groups.append(group_rewards)

    mean_reward = np.mean(all_rewards)
    accuracy = n_correct / max(n_total, 1)
    print(f"  Mean reward: {mean_reward:.4f}")
    print(f"  Accuracy:    {accuracy:.2%} ({n_correct}/{n_total})")

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # PHASE C: LOAD HF MODEL + TRAIN
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    print(f"\n[Phase C] Loading HF model for training...")
    model = load_hf_model_with_lora(MODEL_PATH, lora_for_gen)

    # Take RELEX base snapshot on first round
    if round_idx == 0:
        relex.set_base(model)

    # Set up optimizer
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=GRPO_LR,
        weight_decay=0.01,
    )

    # ── Compute reference + old logprobs ─────────────────────
    print(f"  Computing ref & old logprobs...")
    formatted_prompts = [format_prompt_for_generation(p, tokenizer) for p in prompts]

    # Process in mini-batches to manage memory
    MINI_BATCH = 4
    ref_logprobs_all = []
    old_logprobs_all = []

    for mb_start in range(0, len(prompts), MINI_BATCH):
        mb_end = min(mb_start + MINI_BATCH, len(prompts))
        mb_prompts = formatted_prompts[mb_start:mb_end]
        mb_rollouts = rollouts[mb_start:mb_end]

        for pi, (fp, group) in enumerate(zip(mb_prompts, mb_rollouts)):
            ref_lps_group = []
            old_lps_group = []
            for rollout in group:
                # Old logprobs (LoRA enabled)
                old_lp = compute_per_token_logprobs_batch(
                    model, tokenizer, [fp], [rollout["text"]]
                )[0].detach()

                # Ref logprobs (LoRA disabled)
                model.disable_adapter_layers()
                ref_lp = compute_per_token_logprobs_batch(
                    model, tokenizer, [fp], [rollout["text"]]
                )[0].detach()
                model.enable_adapter_layers()

                ref_lps_group.append(ref_lp)
                old_lps_group.append(old_lp)

            ref_logprobs_all.append(ref_lps_group)
            old_logprobs_all.append(old_lps_group)

    # ── GRPO policy update ───────────────────────────────────
    print(f"  Running GRPO update...")
    grpo_stats = grpo_training_step(
        model, tokenizer, optimizer,
        prompt_texts=prompts,
        rollout_groups=rollouts,
        rewards_groups=rewards_groups,
        ref_logprobs_groups=ref_logprobs_all,
        old_logprobs_groups=old_logprobs_all,
        clip_eps=CLIP_EPSILON,
        kl_coeff=KL_COEFF,
    )
    print(f"  GRPO loss: {grpo_stats['loss']:.4f}")

    # ── SFT regularization ───────────────────────────────────
    print(f"  Running {SFT_STEPS_PER_ROUND} SFT regularization steps...")
    sft_losses = []
    for sft_step_idx in range(SFT_STEPS_PER_ROUND):
        sft_samples = random.sample(all_samples, min(2, len(all_samples)))
        sft_loss = sft_step(model, tokenizer, optimizer, sft_samples)
        sft_losses.append(sft_loss)
    mean_sft_loss = np.mean(sft_losses)
    print(f"  Mean SFT loss: {mean_sft_loss:.4f}")

    # ── Save adapter + RELEX snapshot ────────────────────────
    model.save_pretrained(OUTPUT_DIR)    # always save latest for next vLLM load
    relex.snapshot(model, round_idx)
    ckpt_mgr.maybe_save(model, mean_reward, round_idx)

    # Free model
    free_hf_model(model)
    del optimizer
    gc.collect()
    torch.cuda.empty_cache()

    # ── Log stats ────────────────────────────────────────────
    round_time = time.time() - t_round
    stats_log.append({
        "round": round_idx,
        "mean_reward": mean_reward,
        "accuracy": accuracy,
        "grpo_loss": grpo_stats["loss"],
        "sft_loss": mean_sft_loss,
        "time_min": round_time / 60,
    })
    print(f"\n  Round {round_idx+1} complete in {round_time/60:.1f} min")
    print(f"  {ckpt_mgr.status()}")

print(f"\n{'='*60}")
print(f"  ALL {NUM_ROUNDS} ROUNDS COMPLETE")
print(f"  {ckpt_mgr.status()}")
print(f"{'='*60}")

In [ ]:
# ============================================================
# 13. RELEX EXTRAPOLATION — predict further-trained weights
# ============================================================

# Extrapolate to 2x the actual rounds (e.g., 10 rounds → predict round 20)
TARGET_ROUND = NUM_ROUNDS * 2

predicted_state = relex.extrapolate(target_round=TARGET_ROUND)

if predicted_state is not None:
    # Load model, apply extrapolated weights, save
    print("\nApplying RELEX-extrapolated weights...")
    model = load_hf_model_with_lora(MODEL_PATH, BEST_ADAPTER_DIR)
    relex.apply_to_model(model, predicted_state)

    # Save as a separate adapter
    RELEX_DIR = "/kaggle/working/relex_adapter"
    os.makedirs(RELEX_DIR, exist_ok=True)
    model.save_pretrained(RELEX_DIR)

    size_mb = sum(
        os.path.getsize(os.path.join(RELEX_DIR, f))
        for f in os.listdir(RELEX_DIR)
    ) / 1e6
    print(f"RELEX adapter saved: {RELEX_DIR} ({size_mb:.1f} MB)")
    print("NOTE: Test BOTH best_adapter and relex_adapter to see which scores higher")

    free_hf_model(model)
else:
    print("RELEX extrapolation skipped (not enough snapshots)")

In [ ]:
# ============================================================
# 14. TRAINING SUMMARY & FINAL CHECKS
# ============================================================

print("=" * 60)
print("  GRPO + PRM TRAINING SUMMARY")
print("=" * 60)

# Print stats table
print(f"\n{'Round':>6} {'Reward':>8} {'Accuracy':>10} {'GRPO Loss':>10} {'SFT Loss':>10} {'Time':>8}")
print("-" * 60)
for s in stats_log:
    print(f"{s['round']+1:>6d} {s['mean_reward']:>8.4f} {s['accuracy']:>9.2%} "
          f"{s['grpo_loss']:>10.4f} {s['sft_loss']:>10.4f} {s['time_min']:>7.1f}m")

print(f"\n{ckpt_mgr.status()}")

# Verify adapter files
for adapter_dir, label in [(BEST_ADAPTER_DIR, "BEST"), (OUTPUT_DIR, "LATEST")]:
    if os.path.exists(adapter_dir):
        files = os.listdir(adapter_dir)
        has_config = "adapter_config.json" in files
        has_safetensor = any(f.endswith(".safetensors") for f in files)
        size_mb = sum(os.path.getsize(os.path.join(adapter_dir, f)) for f in files) / 1e6
        print(f"\n  {label} adapter: {adapter_dir}")
        print(f"    adapter_config.json: {'YES' if has_config else 'MISSING'}")
        print(f"    safetensors:         {'YES' if has_safetensor else 'MISSING'}")
        print(f"    Total size:          {size_mb:.1f} MB")

relex_dir = "/kaggle/working/relex_adapter"
if os.path.exists(relex_dir):
    files = os.listdir(relex_dir)
    size_mb = sum(os.path.getsize(os.path.join(relex_dir, f)) for f in files) / 1e6
    print(f"\n  RELEX adapter: {relex_dir} ({size_mb:.1f} MB)")

print("\nDone! Submit the best-performing adapter.")

In [ ]:
# ============================================================
# 15. ZIP ADAPTER FOR SUBMISSION
# ============================================================
import zipfile

# Zip the best adapter
ZIP_PATH = "/kaggle/working/grpo_adapter.zip"
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir(BEST_ADAPTER_DIR):
        fpath = os.path.join(BEST_ADAPTER_DIR, f)
        if os.path.isfile(fpath):
            zf.write(fpath, f)
            print(f"  Added: {f}")

zip_size_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f"\nAdapter zip: {ZIP_PATH} ({zip_size_mb:.1f} MB)")
print("Ready for submission!")